In [1]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 96.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 623.9/623.9 kB 46.3 MB/s eta 0:00:00


In [2]:
import optuna
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the Pima Indian Diabetes dataset (from UCI repository)
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']

# Load the dataset
df = pd.read_csv(url, names=columns)
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
import numpy as np

# Replace zero values with NaN in columns where zero is not a valid value
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

# Impute the missing values with the mean of the respective column
df.fillna(df.mean(), inplace=True)

# Check if there are any remaining missing values
print(df.isnull().sum())


Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [4]:
# Split into features (X) and target (y)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Split data into training and test sets (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Optional: Scale the data for better model performance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Check the shape of the data
print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')


Training set shape: (537, 8)
Test set shape: (231, 8)


In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score


In [6]:
# Create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())
study.optimize(objective, n_trials=20)


[I 2026-08-30 03:40:52,764] A new study created in memory with name: no-name-c34c77b1-5047-4d05-a247-4198dd28bf78
[I 2026-08-30 03:40:53,246] Trial 0 finished with value: 0.7746741154562384 and parameters: {'n_estimators': 134, 'max_depth': 17}. Best is trial 0 with value: 0.7746741154562384.
[I 2026-08-30 03:40:53,801] Trial 1 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 156, 'max_depth': 14}. Best is trial 0 with value: 0.7746741154562384.
[I 2026-08-30 03:40:53,971] Trial 2 finished with value: 0.7579143389199254 and parameters: {'n_estimators': 52, 'max_depth': 4}. Best is trial 0 with value: 0.7746741154562384.
[I 2026-08-30 03:40:54,313] Trial 3 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 95, 'max_depth': 16}. Best is trial 0 with value: 0.7746741154562384.
[I 2026-08-30 03:40:54,587] Trial 4 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 82, 'max_depth': 5}. Best is trial 0 with value: 0.774674115

In [7]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7783985102420857
Best hyperparameters: {'n_estimators': 116, 'max_depth': 20}


In [8]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.74


### ***Samplers in Optuna***

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score


In [10]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.RandomSampler())
study.optimize(objective, n_trials=25)

[I 2026-08-30 03:41:02,195] A new study created in memory with name: no-name-35cad8ee-c864-4e51-bc5f-8ac3fb7ef3e8
[I 2026-08-30 03:41:02,693] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 139, 'max_depth': 11}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-08-30 03:41:02,968] Trial 1 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 76, 'max_depth': 8}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-08-30 03:41:03,298] Trial 2 finished with value: 0.7597765363128491 and parameters: {'n_estimators': 92, 'max_depth': 10}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-08-30 03:41:03,683] Trial 3 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 110, 'max_depth': 8}. Best is trial 3 with value: 0.7709497206703911.
[I 2026-08-30 03:41:04,006] Trial 4 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 87, 'max_depth': 14}. Best is trial 3 with value: 0.770949720

In [11]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7802607076350093
Best hyperparameters: {'n_estimators': 107, 'max_depth': 12}


In [12]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.75


In [13]:
search_space = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [5, 10, 15, 20]
}

In [14]:
# Create a study and optimize it using GridSampler
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.GridSampler(search_space))
study.optimize(objective)

[I 2026-08-30 03:41:14,421] A new study created in memory with name: no-name-3aad7eb3-22d4-445e-8c22-8b7176eb26e5
[I 2026-08-30 03:41:14,758] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 5}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-08-30 03:41:15,293] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 150, 'max_depth': 10}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-08-30 03:41:15,484] Trial 2 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 50, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-08-30 03:41:15,845] Trial 3 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 100, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-08-30 03:41:16,202] Trial 4 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 20}. Best is trial 2 with value: 0.772811

In [15]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7746741154562384
Best hyperparameters: {'n_estimators': 50, 'max_depth': 5}


In [16]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.74


### ***Optuna Visualizations***

In [17]:
# For visualizations
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [18]:
# 1. Optimization History
plot_optimization_history(study).show()

In [19]:
# 2. Parallel Coordinates Plot
plot_parallel_coordinate(study).show()

In [20]:
# 3. Slice Plot
plot_slice(study).show()

In [21]:
# 4. Contour Plot
plot_contour(study).show()

In [22]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()

### ***Optimizing Multiple ML Models***

In [23]:
# Importing the required libraries
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

In [24]:
# Define the objective function for Optuna
def objective(trial):
    # Choose the algorithm to tune
    classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

    if classifier_name == 'SVM':
        # SVM hyperparameters
        c = trial.suggest_float('C', 0.1, 100, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

        model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

    elif classifier_name == 'RandomForest':
        # Random Forest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

    elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score

In [25]:
# Create a study and optimize it using CmaEsSampler
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

[I 2026-08-30 03:41:23,931] A new study created in memory with name: no-name-60179434-c831-4236-b880-eba2e3222642
[I 2026-08-30 03:41:23,947] Trial 0 finished with value: 0.7858472998137803 and parameters: {'classifier': 'SVM', 'C': 0.2020347300491629, 'kernel': 'linear', 'gamma': 'scale'}. Best is trial 0 with value: 0.7858472998137803.
[I 2026-08-30 03:41:24,401] Trial 1 finished with value: 0.7355679702048418 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 82, 'learning_rate': 0.17170532362616256, 'max_depth': 6, 'min_samples_split': 10, 'min_samples_leaf': 10}. Best is trial 0 with value: 0.7858472998137803.
[I 2026-08-30 03:41:24,420] Trial 2 finished with value: 0.7616387337057727 and parameters: {'classifier': 'SVM', 'C': 6.5913065357628975, 'kernel': 'rbf', 'gamma': 'auto'}. Best is trial 0 with value: 0.7858472998137803.
[I 2026-08-30 03:41:25,149] Trial 3 finished with value: 0.7616387337057727 and parameters: {'classifier': 'RandomForest', 'n_estimators': 

In [26]:
# Retrieve the best trial
best_trial = study.best_trial
print("Best trial parameters:", best_trial.params)
print("Best trial accuracy:", best_trial.value)

Best trial parameters: {'classifier': 'SVM', 'C': 0.13137724594772582, 'kernel': 'linear', 'gamma': 'scale'}
Best trial accuracy: 0.7895716945996275


In [27]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,state
0,0,0.785847,2026-08-30 03:41:23.932453,2026-08-30 03:41:23.947581,0 days 00:00:00.015128,0.202035,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
1,1,0.735568,2026-08-30 03:41:23.948076,2026-08-30 03:41:24.401926,0 days 00:00:00.453850,NaN,NaN,GradientBoosting,NaN,NaN,0.171705,6.0,10.0,10.0,82.0,COMPLETE
2,2,0.761639,2026-08-30 03:41:24.402539,2026-08-30 03:41:24.420780,0 days 00:00:00.018241,6.591307,NaN,SVM,auto,rbf,NaN,NaN,NaN,NaN,NaN,COMPLETE
3,3,0.761639,2026-08-30 03:41:24.421261,2026-08-30 03:41:25.149665,0 days 00:00:00.728404,NaN,True,RandomForest,NaN,NaN,NaN,3.0,4.0,9.0,239.0,COMPLETE
4,4,0.767225,2026-08-30 03:41:25.150312,2026-08-30 03:41:25.481656,0 days 00:00:00.331344,NaN,True,RandomForest,NaN,NaN,NaN,18.0,8.0,4.0,103.0,COMPLETE
5,5,0.728119,2026-08-30 03:41:25.482250,2026-08-30 03:41:27.041780,0 days 00:00:01.559530,NaN,NaN,GradientBoosting,NaN,NaN,0.169352,14.0,7.0,2.0,175.0,COMPLETE
6,6,0.750466,2026-08-30 03:41:27.042463,2026-08-30 03:41:27.720469,0 days 00:00:00.678006,NaN,NaN,GradientBoosting,NaN,NaN,0.279112,5.0,6.0,9.0,132.0,COMPLETE
7,7,0.726257,2026-08-30 03:41:27.721061,2026-08-30 03:41:27.735298,0 days 00:00:00.014237,0.336804,NaN,SVM,auto,poly,NaN,NaN,NaN,NaN,NaN,COMPLETE
8,8,0.728119,2026-08-30 03:41:27.735724,2026-08-30 03:41:27.760065,0 days 00:00:00.024341,39.605416,NaN,SVM,auto,rbf,NaN,NaN,NaN,NaN,NaN,COMPLETE
9,9,0.785847,2026-08-30 03:41:27.760625,2026-08-30 03:41:27.780242,0 days 00:00:00.019617,2.494467,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE


In [28]:
study.trials_dataframe()['params_classifier'].value_counts()

,count
params_classifier,
SVM,36
GradientBoosting,7
RandomForest,7


In [29]:
study.trials_dataframe().groupby('params_classifier')['value'].mean()

,value
params_classifier,
GradientBoosting,0.740889
RandomForest,0.768821
SVM,0.772760


In [30]:
# 1. Optimization History
plot_optimization_history(study).show()

In [31]:
# 3. Slice Plot
plot_slice(study).show()

In [32]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()

In [38]:
!pip install xgboost

In [39]:
import optuna
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
import numpy as np

# Install xgboost if not already installed
!pip install xgboost
!pip install optuna-integration[xgboost]

# Load the Iris dataset
X, y = load_iris(return_X_y=True)

# Split the dataset into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the objective function for XGBoost
def objective(trial):
    # Hyperparameter search space
    param = {
        'verbosity': 0,
        'objective': 'multi:softprob',
        'num_class': 3,
        'eval_metric': 'mlogloss',  # Ensure that the eval_metric is specified here
        'booster': 'gbtree',
        'lambda': trial.suggest_float('lambda', 1e-8, 1.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-8, 1.0, log=True),
        'eta': trial.suggest_float('eta', 0.01, 0.3),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'n_estimators': 300,
    }

    # Create DMatrix for XGBoost
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtest = xgb.DMatrix(X_test, label=y_test)

    # Define a pruning callback based on evaluation metrics
    pruning_callback = optuna.integration.XGBoostPruningCallback(trial, "eval-mlogloss")  # Match the metric name in the evals list

    # Train the model
    bst = xgb.train(
        param,
        dtrain,
        num_boost_round=300,
        evals=[(dtrain, "train"), (dtest, "eval")],  # Ensure the eval datasets and names are specified
        early_stopping_rounds=30,
        callbacks=[pruning_callback]
    )

    # Predict on the test set
    preds = bst.predict(dtest)
    best_preds = [int(np.argmax(line)) for line in preds]

    # Return accuracy as the objective value
    accuracy = accuracy_score(y_test, best_preds)
    return accuracy

# Create a study with pruning
study = optuna.create_study(direction='maximize', pruner=optuna.pruners.SuccessiveHalvingPruner())
study.optimize(objective, n_trials=30)

# Output the best trial
print(f"Best trial: {study.best_trial.params}")
print(f"Best accuracy: {study.best_value}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.4/103.4 kB 4.7 MB/s eta 0:00:00


[I 2026-08-30 03:43:14,421] A new study created in memory with name: no-name-b194c55d-8746-4068-8f61-41968daec6f6


[0]	train-mlogloss:1.06617	eval-mlogloss:1.06617
[1]	train-mlogloss:1.02567	eval-mlogloss:1.02288
[2]	train-mlogloss:0.98858	eval-mlogloss:0.98446
[3]	train-mlogloss:0.95663	eval-mlogloss:0.95247
[4]	train-mlogloss:0.92268	eval-mlogloss:0.91676
[5]	train-mlogloss:0.89356	eval-mlogloss:0.88611
[6]	train-mlogloss:0.86656	eval-mlogloss:0.85787
[7]	train-mlogloss:0.84700	eval-mlogloss:0.83788
[8]	train-mlogloss:0.82146	eval-mlogloss:0.81055
[9]	train-mlogloss:0.79425	eval-mlogloss:0.78216
[10]	train-mlogloss:0.76793	eval-mlogloss:0.75335
[11]	train-mlogloss:0.74423	eval-mlogloss:0.72750
[12]	train-mlogloss:0.72521	eval-mlogloss:0.70726
[13]	train-mlogloss:0.70601	eval-mlogloss:0.68782
[14]	train-mlogloss:0.68321	eval-mlogloss:0.66417
[15]	train-mlogloss:0.66305	eval-mlogloss:0.64236
[16]	train-mlogloss:0.64249	eval-mlogloss:0.61976
[17]	train-mlogloss:0.62331	eval-mlogloss:0.59900
[18]	train-mlogloss:0.60470	eval-mlogloss:0.57984
[19]	train-mlogloss:0.58961	eval-mlogloss:0.56438
[20]	train

/usr/lib/python3.13/importlib/__init__.py:88: FutureWarning:

`optuna.integration.xgboost` has been deprecated in v4.9.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.9.0. Use `optuna_integration.xgboost` instead.



[146]	train-mlogloss:0.16235	eval-mlogloss:0.10081
[147]	train-mlogloss:0.16202	eval-mlogloss:0.10042
[148]	train-mlogloss:0.16162	eval-mlogloss:0.09987
[149]	train-mlogloss:0.16145	eval-mlogloss:0.09955
[150]	train-mlogloss:0.16112	eval-mlogloss:0.09914
[151]	train-mlogloss:0.16099	eval-mlogloss:0.09894
[152]	train-mlogloss:0.16072	eval-mlogloss:0.09887
[153]	train-mlogloss:0.16027	eval-mlogloss:0.09831
[154]	train-mlogloss:0.16025	eval-mlogloss:0.09826
[155]	train-mlogloss:0.16028	eval-mlogloss:0.09833
[156]	train-mlogloss:0.15969	eval-mlogloss:0.09745
[157]	train-mlogloss:0.15969	eval-mlogloss:0.09741
[158]	train-mlogloss:0.15933	eval-mlogloss:0.09745
[159]	train-mlogloss:0.15924	eval-mlogloss:0.09732
[160]	train-mlogloss:0.15922	eval-mlogloss:0.09730
[161]	train-mlogloss:0.15919	eval-mlogloss:0.09723
[162]	train-mlogloss:0.15920	eval-mlogloss:0.09725
[163]	train-mlogloss:0.15907	eval-mlogloss:0.09698
[164]	train-mlogloss:0.15906	eval-mlogloss:0.09699
[165]	train-mlogloss:0.15905	ev

...

[0]	train-mlogloss:0.95193	eval-mlogloss:0.94241
[1]	train-mlogloss:0.83356	eval-mlogloss:0.81455


[I 2026-08-30 03:43:17,108] Trial 28 pruned. Trial was pruned at iteration 2.


[0]	train-mlogloss:0.90474	eval-mlogloss:0.88777
[1]	train-mlogloss:0.71459	eval-mlogloss:0.68570


[I 2026-08-30 03:43:17,122] Trial 29 pruned. Trial was pruned at iteration 2.


Best trial: {'lambda': 3.2246298560619254e-07, 'alpha': 1.316660938476552e-07, 'eta': 0.03196587611314471, 'gamma': 0.0004958214429954585, 'max_depth': 9, 'min_child_weight': 4, 'subsample': 0.5130535342673512, 'colsample_bytree': 0.5601174334707635}
Best accuracy: 1.0


In [40]:
! pip install optuna-integration[xgboost]

In [41]:
from optuna.visualization import plot_intermediate_values

# 1. Plot intermediate values during the trials
plot_intermediate_values(study).show()